# 22 — Train-only EDA

**Objetivo:** estudiar distribución, missingness y cardinalidad usando exclusivamente TRAIN. Validation y Test no participan en decisiones exploratorias de features.

In [6]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.train_eda import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-22-train-eda-v1'

In [7]:
from argos.gold.train_eda import build_train_eda

result = build_train_eda(
    silver_dir=paths.silver,
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
)

In [8]:
display(result.target_profile)
display(result.feature_profile)
display(result.category_cardinality)
display(result.numeric_summary)

,train_rows,positives,negatives,null_labels,prevalence,min_t0,max_t0
0,2781601,58446,2723155,0,0.021012,2015-06-11,2023-12-31


,feature,kind,available,missing_rate
0,departamento,categorical,True,0.0
1,ciudad,categorical,True,0.0
2,orden,categorical,True,0.0
3,sector,categorical,True,0.0
4,rama,categorical,True,0.0
5,entidad_centralizada,categorical,True,0.0
6,codigo_de_categoria_principal,categorical,True,0.0
7,tipo_de_contrato,categorical,True,0.0
8,modalidad_de_contratacion,categorical,True,0.0
9,justificacion_modalidad_de_contratacion,categorical,True,0.0


,feature,row_count,missing_rows,cardinality,nonnull_cardinality
0,departamento,2781601,0,34,34
1,ciudad,2781601,0,755,755
2,orden,2781601,0,4,4
3,sector,2781601,0,26,26
4,rama,2781601,0,5,5
5,entidad_centralizada,2781601,0,2,2
6,codigo_de_categoria_principal,2781601,0,11930,11930
7,tipo_de_contrato,2781601,0,22,22
8,modalidad_de_contratacion,2781601,0,15,15
9,justificacion_modalidad_de_contratacion,2781601,0,38,38


,feature,row_count,missing_rows,min_value,median_value,mean_value,max_value


In [9]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,eda_train_only,leakage,True,True,"{""max_t0"": ""2023-12-31 00:00:00"", ""min_t0"": ""2...",EDA population is explicitly restricted to spl...
1,eda_train_labels_complete,target,True,True,"{""null_labels"": 0}",train-only EDA contains no censored/unlabeled ...
2,eda_candidate_features_visible,features,True,True,"{""available_candidates"": 19, ""configured_candi...",at least 5 conservative candidate features are...


In [10]:
blocking = result.validation.loc[result.validation["blocking"]]
print("22 ✅ PASS" if result.ready_for_next_notebook else "22 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Train-only EDA no supera sus checks bloqueantes."

22 ✅ PASS
Blocking checks: 3/3
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_train_eda.json
